In [ ]:
# !pip install python-dotenv

In [1]:
import sys
import os

# Add the project root to the Python path to use the local source code
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
from dotenv import load_dotenv
import os
import guidance
from guidance import models

load_dotenv()

#no need , it is in env file
#os.environ["OPENAI_API_KEY"] = '' # specify your key here

True

In [3]:
from typing import Dict, Tuple, List

sea_ice_variables = [
    "geopotential_heights",
    "relative_humidity",
    "sea_level_pressure",
    "zonal_wind_at_10_meters",
    "meridional_wind_at_10_meters",
    "sensible_plus_latent_heat_flux",
    "total_precipitation",
    "total_cloud_cover",
    "total_cloud_water_path",
    "surface_net_shortwave_flux",
    "surface_net_longwave_flux",
    "northern_hemisphere_sea_ice_extent",
]

treatment = "surface_net_longwave_flux"
outcome = "northern_hemisphere_sea_ice_extent"

# ground truth confounders to the relationship between surface_net_longwave_flux and northern_hemisphere_sea_ice_extent
sea_ice_confounders = ["total_precipitation"]

sea_ice_relationships: List[Tuple[str, str]] = [
    ("surface_net_longwave_flux", "northern_hemisphere_sea_ice_extent"),

    ("geopotential_heights", "surface_net_longwave_flux"),
    ("geopotential_heights", "relative_humidity"),
    ("geopotential_heights", "sea_level_pressure"),

    ("relative_humidity", "total_cloud_cover"),
    ("relative_humidity", "total_cloud_water_path"),
    ("relative_humidity", "total_precipitation"),
    ("relative_humidity", "surface_net_longwave_flux"),

    ("sea_level_pressure", "relative_humidity"),
    ("sea_level_pressure", "geopotential_heights"),
    ("sea_level_pressure", "zonal_wind_at_10_meters"),
    ("sea_level_pressure", "northern_hemisphere_sea_ice_extent"),
    ("sea_level_pressure", "sensible_plus_latent_heat_flux"),
    ("sea_level_pressure", "meridional_wind_at_10_meters"),

    ("zonal_wind_at_10_meters", "northern_hemisphere_sea_ice_extent"),
    ("zonal_wind_at_10_meters", "sensible_plus_latent_heat_flux"),

    ("meridional_wind_at_10_meters", "northern_hemisphere_sea_ice_extent"),
    ("meridional_wind_at_10_meters", "sensible_plus_latent_heat_flux"),

    ("sensible_plus_latent_heat_flux", "northern_hemisphere_sea_ice_extent"),
    ("sensible_plus_latent_heat_flux", "sea_level_pressure"),
    ("sensible_plus_latent_heat_flux", "zonal_wind_at_10_meters"),
    ("sensible_plus_latent_heat_flux", "meridional_wind_at_10_meters"),
    ("sensible_plus_latent_heat_flux", "total_precipitation"),
    ("sensible_plus_latent_heat_flux", "total_cloud_cover"),
    ("sensible_plus_latent_heat_flux", "total_cloud_water_path"),

    ("total_precipitation", "northern_hemisphere_sea_ice_extent"),
    ("total_precipitation", "relative_humidity"),
    ("total_precipitation", "sensible_plus_latent_heat_flux"),
    ("total_precipitation", "surface_net_longwave_flux"),
    ("total_precipitation", "total_cloud_cover"),
    ("total_precipitation", "total_cloud_water_path"),

    ("total_cloud_water_path", "total_precipitation"),
    ("total_cloud_water_path", "sensible_plus_latent_heat_flux"),
    ("total_cloud_water_path", "relative_humidity"),
    ("total_cloud_water_path", "surface_net_longwave_flux"),
    ("total_cloud_water_path", "surface_net_shortwave_flux"),

    ("total_cloud_cover", "total_precipitation"),
    ("total_cloud_cover", "sensible_plus_latent_heat_flux"),
    ("total_cloud_cover", "relative_humidity"),
    ("total_cloud_cover", "surface_net_longwave_flux"),
    ("total_cloud_cover", "surface_net_shortwave_flux"),

    ("surface_net_shortwave_flux", "northern_hemisphere_sea_ice_extent"),

    ("northern_hemisphere_sea_ice_extent", "sea_level_pressure"),
    ("northern_hemisphere_sea_ice_extent", "zonal_wind_at_10_meters"),
    ("northern_hemisphere_sea_ice_extent", "meridional_wind_at_10_meters"),
    ("northern_hemisphere_sea_ice_extent", "sensible_plus_latent_heat_flux"),
    ("northern_hemisphere_sea_ice_extent", "surface_net_shortwave_flux"),
    ("northern_hemisphere_sea_ice_extent", "surface_net_longwave_flux"),
]

## Helpers

Model type - the type of LLM used
By default it's set to completions models

Relationship strategy - is the type of request made to the LLM (request parent, child, pairwise relationship)

In [4]:
from pywhyllm import ModelType, RelationshipStrategy
model_type = ModelType.Completion
relationship_strategy = RelationshipStrategy.Parent

## Model

In [8]:
from pywhyllm.suggesters.model_suggester import ModelSuggester
m = ModelSuggester('gpt-4o-mini')


In [9]:
domain_expertises = m.suggest_domain_expertises(sea_ice_variables)

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

In [10]:
domain_expertises

['Climate Scientists']

In [11]:
domain_experts = m.suggest_domain_experts(sea_ice_variables)

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

In [12]:
domain_experts

{'Atmospheric Scientist',
 'Climatologist',
 'Meteorologist',
 'Oceanographer',
 'Statistical Modeler/Quantitative Analyst'}

In [13]:
parents = m.suggest_parents(domain_expertises[0], "relative_humidity", sea_ice_variables)

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

In [14]:
parents

[]

In [15]:
children = m.suggest_children(domain_expertises[0], "relative_humidity", sea_ice_variables)

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

In [16]:
children

['total_cloud_cover', 'total_precipitation', 'sensible_plus_latent_heat_flux']

In [17]:
pairwise_relationship = m.suggest_pairwise_relationship(domain_expertises[0], "total_precipitation", "relative_humidity")

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

In [18]:
pairwise_relationship

('total_precipitation', 'relative_humidity')

In [19]:
confounders = m.suggest_confounders(treatment, outcome, sea_ice_variables, domain_expertises)

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

In [20]:
confounders

({('surface_net_longwave_flux', 'northern_hemisphere_sea_ice_extent'): 1,
  ('total_cloud_cover', 'surface_net_longwave_flux'): 1,
  ('total_cloud_cover', 'northern_hemisphere_sea_ice_extent'): 1},
 ['total_cloud_cover'])

In [21]:
"""returns a dictionary with the how many times that edge was suggested"""
model_edges = m.suggest_relationships(treatment, outcome, sea_ice_variables, domain_expertises, RelationshipStrategy.Pairwise)

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

In [22]:
model_edges

{('geopotential_heights', 'relative_humidity'): 1,
 ('geopotential_heights', 'sea_level_pressure'): 1,
 ('geopotential_heights', 'zonal_wind_at_10_meters'): 1,
 ('geopotential_heights', 'meridional_wind_at_10_meters'): 1,
 ('geopotential_heights', 'sensible_plus_latent_heat_flux'): 1,
 ('geopotential_heights', 'total_precipitation'): 1,
 ('geopotential_heights', 'total_cloud_cover'): 1,
 ('geopotential_heights', 'total_cloud_water_path'): 1,
 ('geopotential_heights', 'surface_net_shortwave_flux'): 1,
 ('geopotential_heights', 'surface_net_longwave_flux'): 1,
 ('geopotential_heights', 'northern_hemisphere_sea_ice_extent'): 1,
 ('sea_level_pressure', 'relative_humidity'): 1,
 ('meridional_wind_at_10_meters', 'relative_humidity'): 1,
 ('relative_humidity', 'sensible_plus_latent_heat_flux'): 1,
 ('relative_humidity', 'total_precipitation'): 1,
 ('relative_humidity', 'total_cloud_cover'): 1,
 ('relative_humidity', 'total_cloud_water_path'): 1,
 ('relative_humidity', 'surface_net_shortwave_f

## Identifier

In [ ]:
from pywhyllm.suggesters.identification_suggester import IdentificationSuggester
i = IdentificationSuggester('gpt-4o-mini')

In [ ]:
"""calls modeler suggest_confounders in the background"""
backdoor = i.suggest_backdoor(treatment, outcome, sea_ice_variables, domain_expertises, "causal mechanisms")

In [ ]:
backdoor

In [ ]:
"""suggests instrumental variables"""
ivs = i.suggest_ivs(treatment, outcome, sea_ice_variables, domain_expertises)

In [ ]:
ivs

## Validator

In [ ]:
from pywhyllm.suggesters.validation_suggester import ValidationSuggester
v = ValidationSuggester('gpt-4')

In [ ]:
latent_confounders = v.suggest_latent_confounders(treatment, outcome, domain_expertises)

In [ ]:
latent_confounders

In [ ]:
negative_controls = v.suggest_negative_controls(treatment, outcome, sea_ice_variables, domain_expertises)

In [ ]:
negative_controls

In [ ]:
critique = v.request_pairwise_critique(domain_expertises[0], "total_precipitation", "relative_humidity")

In [ ]:
critique

In [ ]:
parent=RelationshipStrategy.Parent
child=RelationshipStrategy.Child
pairwise=RelationshipStrategy.Pairwise

In [ ]:
critique = v.critique_graph(sea_ice_variables, model_edges, domain_expertises, pairwise)

In [ ]:
critique